# Deformetrica atlas workflow for weevil trochanters

This output-free notebook orchestrates the study-specific deterministic-atlas workflow. It accompanies the command-line scripts in this directory and records the exact final parameters used for the 68-specimen analysis.

The workflow was developed from the *Landmark-Free Morphometry* tutorial by Toussaint and colleagues (GitLab commit `cc10f96`), but contains only the study-specific implementation and configuration.

## 1. Select the working directory

The directory must contain the 68 aligned VTK meshes and `initial_template.vtk`. Raw and processed meshes are distributed separately from the code repository.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPOSITORY_ROOT = Path.cwd().resolve()
WORKFLOW_DIR = REPOSITORY_ROOT / "scripts" / "analysis" / "deformetrica_workflow"
if not WORKFLOW_DIR.is_dir():
    WORKFLOW_DIR = REPOSITORY_ROOT
WORKDIR = Path("data/atlasing").expanduser().resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
print(WORKDIR)

## 2. Install the exact configuration

Set `USE_ARCHIVED_SUBJECT_ORDER` to `True` to reproduce the ordered 68-subject dataset used in the manuscript. Set it to `False` to generate a new dataset XML from all matching aligned meshes.

In [ ]:
USE_ARCHIVED_SUBJECT_ORDER = True

for name in ("model.xml", "optimization_parameters.xml"):
    shutil.copy2(WORKFLOW_DIR / "config" / name, WORKDIR / name)

if USE_ARCHIVED_SUBJECT_ORDER:
    shutil.copy2(WORKFLOW_DIR / "study_data_set.xml", WORKDIR / "data_set.xml")
else:
    subprocess.run(
        [
            sys.executable,
            str(WORKFLOW_DIR / "build_dataset_xml.py"),
            str(WORKDIR),
            "--output",
            "data_set.xml",
        ],
        check=True,
    )

## 3. Run the deterministic atlas

This is the computationally intensive step. It requires Deformetrica 4.3.0 on the active environment's `PATH`. The helper streams output to the notebook and writes `deformetrica.log`, `convergence.txt`, `Atlas_Momentas.txt`, `Atlas_ControlPoints.txt` and `Atlas_initial_template.vtk`.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(WORKFLOW_DIR / "run_deformetrica_atlas.py"),
        str(WORKDIR),
    ],
    check=True,
)

## 4. Verify the final dimensions and convergence

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

header = (WORKDIR / "Atlas_Momentas.txt").read_text(encoding="utf-8").splitlines()[0]
n_subjects, n_control_points, dimension = map(int, header.split())
print(
    f"subjects={n_subjects}, control_points={n_control_points}, "
    f"dimension={dimension}"
)
assert (n_subjects, n_control_points, dimension) == (68, 900, 3)

convergence = np.loadtxt(WORKDIR / "convergence.txt")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(convergence[:, 0], label="log-likelihood")
ax.plot(convergence[:, 1], label="attachment")
ax.plot(convergence[:, 2], label="regularity")
ax.set_xlabel("Iteration")
ax.legend()
fig.tight_layout()

## 5. Derive PCA scores from subject momenta

The R workflow preserves the subject order from `data_set.xml`, reshapes momenta in Deformetrica's subject-control point-dimension order, and performs centred, unscaled PCA.

In [ ]:
PCA_SCRIPT = (
    WORKFLOW_DIR.parent
    / "atlas_pca_workflow"
    / "r"
    / "extract_pca_scores_with_specimen_ids.R"
)
PCA_OUTPUT = WORKDIR / "pca_outputs"
PCA_OUTPUT.mkdir(exist_ok=True)

subprocess.run(
    [
        "Rscript",
        str(PCA_SCRIPT),
        str(WORKDIR / "Atlas_Momentas.txt"),
        str(WORKDIR / "data_set.xml"),
        str(PCA_OUTPUT),
    ],
    check=True,
)

## Outputs

The principal downstream files are `PCA_scores_with_specimen_id.csv` and `PCA_variance.csv` in `pca_outputs`. The first five PCs were retained for the principal downstream analyses described in the manuscript.